In [ ]:
import dash
from dash import html, dcc, dash_table, Input, Output
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import calendar

# -------------------------------- Data load -----------------------------------------------
df = pd.read_csv("C:/Users/Royer/Desktop/python avancé/cours-m1-ecap/datasets/data.csv")
df.dropna(inplace=True)
df['CustomerID'] = df['CustomerID'].astype(int)
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Total_prices'] = df['Avg_Price'] * (1 - df['Discount_pct']/100)

# CA unitaire
df['CA'] = df['Quantity'] * df['Total_prices']

# -------------------------------- Fonctions métiers-------------------------
def frequence_meilleurs_ventes(df):
    x = df.groupby(['Product_Category','Gender'])['Quantity'].sum().reset_index()
    return x.sort_values(by='Quantity', ascending=False).head(10)

def indicateur_mois(df, mois):
    mask = (df['Month'] == mois)
    mois_prec = 12 if mois == 1 else mois-1
    mask_prec = (df['Month'] == mois_prec)
    return {
        "mois": calendar.month_name[mois],
        "ventes": {
            "value": round(df.loc[mask,'Quantity'].sum(),2),
            "reference": round(df.loc[mask_prec,'Quantity'].sum(),2)
        },
        "ca": {
            "value": round(df.loc[mask,'CA'].sum(),2),
            "reference": round(df.loc[mask_prec,'CA'].sum(),2)
        }
    }

# -------------------------------- Dash App ---------------------------------------------
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([

    # HEADER (Fixe)
    dbc.Row([
        dbc.Col(html.H4("ECAP Store - Dashboard", className="text-dark m-0"), md=6),
        dbc.Col(
            dcc.Dropdown(
                id='drop-1',
                options=[{'label': i, 'value': i} for i in df['Location'].unique()],
                placeholder="Filtrer par localisation",
                clearable=True
            ), md=6
        )
    ], style={"height": "8vh", "backgroundColor": "#cfe2ff", "padding": "10px"}, className="align-items-center"),

    # MAIN CONTENT (92% )
    dbc.Row([

        # COLONNE GAUCHE
        dbc.Col([
            # Zone KPI (30% de la hauteur de la colonne)
            dbc.Row([
                dbc.Col(dcc.Graph(id="kpi-ca", style={"height": "100%"}), md=6),
                dbc.Col(dcc.Graph(id="kpi-ventes", style={"height": "100%"}), md=6),
            ], style={"height": "30%"}),

            # Zone Barplot (70% de la hauteur)
            dbc.Row([
                dbc.Col(dcc.Graph(id="graph-top10", style={"height": "100%"}))
            ], style={"height": "70%"}),
        ], md=5, className="vh-90 d-flex flex-column"),

        # COLONNE DROITE
        dbc.Col([
            # Zone Line Chart (50% de la hauteur)
            dbc.Row([
                dbc.Col(dcc.Graph(id="graph-evo", style={"height": "100%"}))
            ], style={"height": "50%"}),

            # Zone Table (50% de la hauteur)
            dbc.Row([
                dbc.Col(
                    dash_table.DataTable(
                        id="table-top100",
                        page_size=10,
                        style_table={"height": "100%", "overflowY": "auto"},
                        style_cell={"padding": "5px", "fontSize": "12px"},
                        fixed_rows={"headers": True}
                    ), className="h-100"
                )
            ], style={"height": "45%", "marginTop": "2%"}) # 45% 
        ], md=7, className="vh-90 d-flex flex-column"),

    ], style={"height": "90vh", "paddingTop": "10px"})

], fluid=True, style={"height": "100vh"})

# ---------------- Callback  ----------------
@app.callback(
    Output("kpi-ca", "figure"),
    Output("kpi-ventes", "figure"),
    Output("graph-top10", "figure"),
    Output("graph-evo", "figure"),
    Output("table-top100", "data"),
    Input("drop-1", "value")
)
def update_dashboard(location):
    dff = df if location is None else df[df["Location"]==location]
    res = indicateur_mois(dff, 12)

    # Communs pour les graphiques pour éviter les marges énormes
    layout_args = dict(margin=dict(t=40, b=20, l=20, r=20), template="simple_white")

    # KPI CA
    fig_kpi_ca = go.Figure(go.Indicator(
        mode="number+delta",
        value=res["ca"]["value"],
        delta={"reference": res["ca"]["reference"]},
        title={"text": res["mois"]}
    ))
    fig_kpi_ca.update_layout(margin=dict(t=0, b=0, l=0, r=0), height=150)

    # KPI Ventes
    fig_kpi_ventes = go.Figure(go.Indicator(
        mode="number+delta",
        value=res["ventes"]["value"],
        delta={"reference": res["ventes"]["reference"]},
        title={"text":res['mois']}
    ))
    fig_kpi_ventes.update_layout(margin=dict(t=0, b=0, l=0, r=0), height=150)

    # Barplot
    data_top10 = frequence_meilleurs_ventes(dff)
    fig_top10 = px.bar(data_top10, x="Quantity", y="Product_Category", color="Gender", 
                       barmode="group", title="Top 10 Ventes")
    fig_top10.update_layout(**layout_args)

    # Line Chart
    evo_ca = dff.groupby(pd.Grouper(key='Transaction_Date', freq='W'))['CA'].sum().reset_index()
    fig_evo = px.line(evo_ca, x="Transaction_Date", y="CA", title="Evolution du Chiffres d'affaires par semaine")
    fig_evo.update_layout(**layout_args)

    # Table
    top100 = dff[['Transaction_Date','Gender','Location','Product_Category','Quantity','Avg_Price','Discount_pct']]
    table_data = top100.sort_values(by='Transaction_Date', ascending=False).head(100).to_dict("records")

    return fig_kpi_ca, fig_kpi_ventes, fig_top10, fig_evo, table_data

if __name__ == "__main__":
    app.run_server(debug=True, port=8053,jupyter_mode="external")

Dash app running on http://127.0.0.1:8053/


c:\Users\Royer\miniconda3.1\envs\dash.env\lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.

c:\Users\Royer\miniconda3.1\envs\dash.env\lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.

c:\Users\Royer\miniconda3.1\envs\dash.env\lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.

c:\Users\Royer\miniconda3.1\envs\dash.env\lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length